##### EDA para explorar estructura del dataset, valores nulos y Análisis de distribución de las columnas 

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

In [ ]:
Path.cwd()

In [ ]:
os.getcwd()

#### Load data

In [ ]:
df = pd.read_csv('../data/raw/HousingData.csv')

#### Estructura del dataset

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.describe()

#### Verificación de Datatypes y porcentaje de valores nulos por columna

In [ ]:

df.info()
df.isnull().sum()
df.isnull().sum()/len(df)
df.isnull().sum()/len(df)


#### Número de valores nulos por columna

In [ ]:
# Count null values in each column
df.isnull().sum()

In [ ]:
nulls = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_pct": (
        df.isnull().mean() * 100
    ).round(2)
})

nulls.sort_values(
    by="null_pct",
    ascending=False,
)

#### Verificación de número de registros duplicados

In [ ]:
df.duplicated().sum()

In [ ]:
df.hist(
    figsize=(15, 12),
    bins=30,
)

plt.tight_layout()
plt.show()

In [ ]:
df.skew(numeric_only=True).sort_values(
    ascending=False
)

In [ ]:
# Columns with missing values
cols_with_nulls = [
    "CRIM",
    "ZN",
    "INDUS",
    "CHAS",
    "AGE",
    "LSTAT",
]

# Plot histograms
df[cols_with_nulls].hist(
    figsize=(12, 8),
    bins=30,
)

plt.tight_layout()
plt.show()

In [ ]:
df[cols_with_nulls].skew()

#### Conteo de Frecuencias

In [ ]:
for col in df.columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head())

In [ ]:
df["CHAS"].value_counts()

#### Matriz de Correlación

In [ ]:
corr = df.corr(numeric_only=True)

corr["MEDV"].sort_values(
    ascending=False
)

In [ ]:
corr[13:]

#### Mapa de Calor

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))


plt.imshow(corr)

plt.colorbar()
plt.xticks(
    range(len(corr.columns)),
    corr.columns,
    rotation=90,
)
plt.yticks(
    range(len(corr.columns)),
    corr.columns,
)

plt.title("Correlation Matrix")
plt.show()

#### Estrategia de Preprocesamiento

In [ ]:
preprocessing_strategy = {
    "CRIM": "median imputation + log1p transform",
    "ZN": "median imputation + log1p transform",
    "INDUS": "median imputation",
    "AGE": "median imputation",
    "LSTAT": "median imputation",
    "CHAS": "mode imputation",
}

preprocessing_strategy

#### Copia EDA con preprocesamiento

In [ ]:
eda_df = df.copy()

# CHAS -> mode
eda_df["CHAS"] = eda_df["CHAS"].fillna(
    eda_df["CHAS"].mode()[0]
)

# CRIM -> median + log1p
eda_df["CRIM"] = eda_df["CRIM"].fillna(
    eda_df["CRIM"].median()
)

eda_df["CRIM_LOG"] = np.log1p(
    eda_df["CRIM"]
)

# ZN -> median + log1p
eda_df["ZN"] = eda_df["ZN"].fillna(
    eda_df["ZN"].median()
)

eda_df["ZN_LOG"] = np.log1p(
    eda_df["ZN"]
)

# Median columns
for col in ["INDUS", "AGE", "LSTAT"]:
    eda_df[col] = eda_df[col].fillna(
        eda_df[col].median()
    )

# Create 5 equal-width bins
# Fill AGE temporarily for feature engineering
eda_df["AGE"] = eda_df["AGE"].fillna(
    eda_df["AGE"].median()
)

# Create 5 equal-width bins
eda_df["AGE_GROUP"] = pd.cut(
    eda_df["AGE"],
    bins=5,
)

# Replace bins with median AGE inside each bin
age_bin_medians = (
    eda_df.groupby("AGE_GROUP")["AGE"]
    .median()
)

eda_df["AGE_BIN"] = (
    eda_df["AGE_GROUP"]
    .map(age_bin_medians)
)

eda_df.head()

#### Verificar que no hay valores nulos

In [ ]:
eda_df[cols_with_nulls].isnull().sum()

#### Comparación Log Transforms

In [ ]:
fig, ax = plt.subplots(
    2,
    2,
    figsize=(12, 8),
)

ax[0, 0].hist(eda_df["CRIM"], bins=30)
ax[0, 0].set_title("CRIM after imputation")

ax[0, 1].hist(eda_df["CRIM_LOG"], bins=30)
ax[0, 1].set_title("CRIM after log1p")

ax[1, 0].hist(eda_df["ZN"], bins=30)
ax[1, 0].set_title("ZN after imputation")

ax[1, 1].hist(eda_df["ZN_LOG"], bins=30)
ax[1, 1].set_title("ZN after log1p")

plt.tight_layout()
plt.show()

In [ ]:
eda_df.corr(numeric_only=True)

In [ ]:
target = "MEDV"

corr_with_target = eda_df.corr(
    numeric_only=True
)[target].sort_values(ascending=False)

corr_with_target

#### Detección de Valores Atípicos

In [ ]:
eda_df.boxplot(
    figsize=(15, 8),
    rot=90,
)

plt.show()